# Clase 12 — Tuning de hiperparámetros con MLflow y Databricks

**Módulo:** Del modelo al producto: diseño y trazabilidad MLOps

**Intención:** usar varios intentos ordenados para mejorar un candidato y conservar su evidencia en un servidor remoto de MLflow.

En las clases anteriores registramos Linear Regression y Random Forest en un servidor local y usamos evidencia para decidir cuál debía ocupar los aliases `champion` y `challenger`. Ahora mantendremos ambos modelos y usaremos **Optuna** para ajustar sus hiperparámetros en el servidor de MLflow de Databricks.

No cambiaremos el problema, las cinco features ni la métrica. El cambio importante es cómo organizamos muchos intentos y dónde conservamos su evidencia.


## Antes de comenzar: actualiza el repositorio

Desde la raíz del repositorio público:

```bash
git status
```

Si no hay cambios pendientes y estás en `main`:

```bash
git pull --ff-only origin main
uv sync --locked
```

`uv sync --locked` reconstruye el ambiente con las dependencias declaradas por el profesor. No ejecutes `uv add` ni `uv lock` en el repositorio público.

En esta clase también crearemos una cuenta individual de **Databricks Free Edition**, reconoceremos su workspace y configuraremos el acceso local. No se presupone que ya tengas la cuenta. No escribas tokens, contraseñas, correos personales ni la URL completa de tu workspace dentro del notebook.


## Punto de partida

**Prerrequisitos concretos:** distinguir experiment, run, parámetro, métrica, tag y artifact; reconocer un parent run y sus child runs; entender que Model Registry organiza versiones y aliases; y poder explicar el pipeline de NYC Taxi.

**Resultado observable:** un experimento exclusivo en Databricks con dos parent runs, sus child runs de Optuna, un modelo final de cada familia evaluado con abril de 2026, dos versiones remotas identificadas como `champion` y `challenger`, y predicciones obtenidas al cargar ambos aliases.

El recorrido conserva:

- marzo de 2026 como datos de desarrollo;
- abril de 2026 como validación final;
- `distancia_km`, `pasajeros`, `hora_recoleccion`, `zona_origen` y `zona_destino`;
- `duracion_minutos` como target;
- RMSE como métrica principal;
- un `Pipeline` que incluye preprocesamiento y modelo.


# 1. Configuraciones comunes de MLflow Tracking

La misma API de MLflow puede guardar información en distintos destinos. Antes de conectarnos a Databricks, comparemos las tres configuraciones que aparecen en la documentación actual.

![Diagrama oficial de MLflow que compara almacenamiento local, tracking local con base de datos y tracking remoto mediante un servidor.](https://mlflow.org/docs/latest/assets/images/tracking-setup-overview-3d8cfd511355d9379328d69573763331.png)

*Fuente: [MLflow Architecture Overview — Common Setups](https://mlflow.org/docs/latest/self-hosting/architecture/overview/), consultada en septiembre de 2026.*

| Configuración | Caso de uso | Metadata | Artifacts | Servidor |
|---|---|---|---|---|
| 1. Localhost | exploración individual | `mlruns/` | `mlruns/` | no es necesario |
| 2. Tracking local con base de datos | trabajo individual más estructurado | SQLite u otra base compatible | sistema de archivos local | opcional |
| 3. Tracking remoto | colaboración | almacenamiento administrado | almacenamiento compartido | sí |

Una precisión importante: el SDK puede trabajar directamente con `mlruns/`, pero el comando actual `mlflow server` usa SQLite por defecto. En las clases 10 y 11 escribimos ambas ubicaciones explícitamente para que siempre fueran visibles.


## 1.1 Lo que ya construimos localmente

![Diagrama histórico de la Clase 10 que muestra el registro de evidencia en MLflow y su consulta en la interfaz.](../assets/modulo-02-ciclo-mlops/clase-10/mlflow-tracking-basics.png)

En las clases 10 y 11 usamos esta arquitectura:

```text
Notebook o script
└── MLflow Python SDK
    └── HTTP → Tracking Server en 127.0.0.1:5000
        ├── metadata → mlflow.db
        ├── artifacts → mlartifacts/
        └── consultas → Tracking UI
```

- El **Backend Store** conserva experiments, runs, parámetros, métricas y tags.
- El **Artifact Store** conserva modelos y otros archivos.
- El **Tracking Server** recibe las operaciones del cliente.
- La **Tracking UI** permite consultar y comparar el estado guardado.


## 1.2 Qué cambia cuando el servidor es remoto

| Permanece igual | Cambia |
|---|---|
| datos, features y target | Tracking URI |
| pipeline y modelos | autenticación |
| parámetros y métricas | ubicación de metadata y artifacts |
| llamadas principales de MLflow | permisos y espacio de nombres |
| criterio de comparación | posibilidad de colaborar |

Cambiar la Tracking URI no copia los runs que ya existen en `mlflow.db`. El servidor local y el servidor de Databricks mantienen estados distintos.


# 2. Del servidor local al remoto

En las clases 10 y 11, los runs quedaron en nuestra computadora. En esta clase enviaremos el mismo tipo de información al servidor administrado de MLflow de Databricks para:

- consultar los trials desde el workspace;
- comparar el champion y el challenger;
- registrar los modelos seleccionados en Unity Catalog.

El entrenamiento seguirá ejecutándose localmente.


# 3. Databricks

Databricks ofrece un workspace remoto y un servidor administrado de MLflow. En esta clase lo usaremos para dos tareas concretas:

1. conservar y comparar runs de tuning;
2. registrar el modelo elegido en Unity Catalog.

![Logotipo de Databricks usado en el material histórico de la clase.](https://upload.wikimedia.org/wikipedia/commons/6/63/Databricks_Logo.png)

No utilizaremos Databricks para desplegar la aplicación final ni para cambiar el modelo conductor. El código seguirá ejecutándose localmente; sólo cambia el destino de Tracking y Registry.

Consultas:

- [MLflow on Databricks](https://docs.databricks.com/aws/en/mlflow)
- [Choose where MLflow data is stored](https://docs.databricks.com/aws/en/mlflow/tracking-server-configuration)
- [MLflow Model Registry in Unity Catalog](https://docs.databricks.com/aws/en/mlflow/models)


## 3.1 Elegir la edición correcta

Para este curso usaremos **Databricks Free Edition para uso personal**. No elijas la prueba gratuita para empresas.

| Opción | Para quién | Duración y pago | ¿La usaremos? |
|---|---|---|---|
| Free Edition | estudiantes y aprendizaje personal | sin costo, sin tarjeta y sujeta a límites de uso | **sí** |
| Free trial | evaluación empresarial | créditos por 14 días; puede convertirse en una cuenta de pago si se configura facturación | no |

Free Edition crea un workspace serverless y almacenamiento predeterminado. No pide elegir AWS, Azure o Google Cloud ni crear una cuenta en esos proveedores. La antigua **Community Edition fue retirada en 2025**; aunque encuentres un tutorial que la mencione, no sigas ese recorrido.

Free Edition es para aprendizaje y uso no comercial. Trabaja únicamente con los datos didácticos del curso: no cargues información sensible, privada o de una organización.

Consultas oficiales:

- [Comparar Free Edition y free trial](https://docs.databricks.com/aws/en/getting-started/free-trial-vs-free-edition)
- [Databricks Free Edition](https://docs.databricks.com/aws/en/getting-started/free-edition)
- [Página de registro de Free Edition](https://login.databricks.com/signup?provider=DB_FREE_TIER)


## 3.2 Crear la cuenta gratuita

1. Abre la [página oficial de Databricks Free Edition](https://login.databricks.com/signup?provider=DB_FREE_TIER).
2. Confirma que la página diga **For personal use** y **Free Edition**. No selecciones la prueba para trabajo o empresas.
3. Continúa con Google, Microsoft o un correo personal. Si eliges correo, completa la verificación que Databricks envíe.
4. Lee y acepta los términos y el aviso de privacidad.
5. Completa los datos que solicite el formulario. No copies esos datos en el repositorio ni en las evidencias del curso.
6. Espera a que Databricks cree automáticamente el workspace. No selecciones proveedor de nube, región, tarjeta ni plan de pago.
7. Si ya tenías Community Edition, crea o migra tu acceso a Free Edition; la edición anterior ya no es la plataforma vigente.

La interfaz puede cambiar ligeramente. La comprobación importante no es el texto exacto de cada botón: al terminar debes entrar a un **workspace**, no a una pantalla de contratación o de configuración de AWS, Azure o Google Cloud.


## 3.3 Reconocer el workspace y guardar su URL

Un **workspace** es el espacio donde Databricks organiza notebooks, experiments, modelos y otros recursos. En Free Edition, cada cuenta del curso tendrá su propio workspace. Esa separación permite que todos usemos los mismos nombres sin mezclar resultados.

Cuando estés dentro del workspace:

1. identifica en la barra lateral **Workspace**, **Experiments** o la búsqueda de recursos;
2. abre **Catalog** o **Catalog Explorer**;
3. identifica el catálogo del workspace y expándelo hasta encontrar el esquema `default`;
4. anota el nombre exacto del catálogo, porque normalmente comparte el nombre del workspace y no tiene por qué llamarse literalmente `workspace`;
5. copia desde el navegador sólo la parte base de la dirección, con una forma semejante a:

```text
https://<identificador-del-workspace>.cloud.databricks.com
```

Guarda temporalmente la URL y el nombre del catálogo fuera del notebook. La dirección será `<url-del-workspace>` en el login de la sección siguiente; el nombre del catálogo se usará después para registrar el modelo. No uses la URL de `accounts.cloud.databricks.com`, una URL de documentación ni una dirección con la ruta de un notebook abierto.

### ✅ Checkpoint: cuenta y workspace

- [ ] Elegí **Free Edition para uso personal**, no free trial.
- [ ] Puedo volver a iniciar sesión en mi workspace.
- [ ] Veo el área de experiments y Catalog Explorer.
- [ ] Identifiqué el catálogo del workspace y su esquema `default`.
- [ ] Conservo la URL base del workspace fuera de Git.
- [ ] No registré tarjeta ni conecté una cuenta de nube.


## 3.4 Límites que sí afectan esta práctica

Free Edition no es un ambiente de producción:

- ofrece un solo workspace y un solo metastore por cuenta;
- utiliza únicamente cómputo serverless y aplica cuotas de uso;
- si se agota una cuota, el cómputo puede quedar suspendido hasta que se restablezca; los datos y la configuración no se borran por ese motivo;
- algunas funciones empresariales y administrativas no están disponibles;
- una cuenta inactiva durante mucho tiempo puede eliminarse;
- no ofrece soporte ni un acuerdo de nivel de servicio.

Para esta clase el entrenamiento se ejecuta en la computadora y Databricks recibe la evidencia de MLflow. No necesitamos crear clusters, habilitar GPU ni configurar infraestructura de nube. Si una cuota o servicio no está disponible, conservaremos el recorrido local de las clases 10–11 y retomaremos el registro remoto después.

Consulta: [Databricks Free Edition limitations](https://docs.databricks.com/aws/en/getting-started/free-edition-limitations).


## 3.5 Crear un Personal Access Token

1. En el workspace, abre tu perfil y selecciona **Settings**.
2. Entra a **Developer**.
3. Junto a **Access tokens**, selecciona **Manage**.
4. Selecciona **Generate new token**.
5. Usa una descripción reconocible, por ejemplo `pcd-otono-2026`.
6. Define una vigencia que cubra el resto del curso.
7. Genera el token y cópialo: Databricks sólo lo muestra una vez.

El token da acceso a tu workspace. No lo pegues en una celda, no lo compartas y no lo subas a Git. Si se expone, revócalo y crea otro.

Consulta: [Databricks personal access tokens](https://docs.databricks.com/aws/en/dev-tools/auth/pat).


## 3.6 Guardar las credenciales en `.env`

En la raíz del repositorio público crea un archivo llamado `.env`:

```dotenv
DATABRICKS_HOST=https://<identificador-del-workspace>.cloud.databricks.com
DATABRICKS_TOKEN=<token-copiado-una-sola-vez>
```

- `DATABRICKS_HOST` es la URL base del workspace.
- `DATABRICKS_TOKEN` es el Personal Access Token.
- No agregues comillas ni espacios alrededor de `=`.
- El `.gitignore` del curso ya excluye `.env`; confirma con `git check-ignore .env`.
- Nunca muestres el contenido con `cat`, una captura o una salida del notebook.

El paquete `python-dotenv` leerá el archivo y cargará esas dos variables sólo en el proceso de Python. El archivo permanece en tu computadora y no forma parte del repositorio.

Consultas: [variables de autenticación de Databricks](https://docs.databricks.com/aws/en/dev-tools/auth/env-vars) y [python-dotenv](https://bbc2.github.io/python-dotenv/).


## 3.7 Antes de continuar

Hasta aquí sólo preparamos el acceso. Antes de abrir Python confirma que tienes, fuera del notebook:

- la URL base del workspace;
- un token vigente;
- ambas variables dentro de `.env`;
- el archivo `.env` ignorado por Git.

El siguiente paso será conectar MLflow y crear el experimento de esta clase.


### Del acceso a la conexión

La cuenta, la URL y el token no producen runs por sí solos. Ahora configuraremos el cliente de MLflow para que cada operación apunte al workspace remoto.


# 4. Conectar Python con Databricks

`load_dotenv()` lee el archivo `.env` y agrega `DATABRICKS_HOST` y `DATABRICKS_TOKEN` como variables de entorno del proceso de Python. Cuando configuramos `databricks` como Tracking URI, el SDK de Python busca automáticamente esas variables: usa el host para saber a qué workspace conectarse y el token para autenticarse. No necesitamos pasar las credenciales en cada llamada.

El token permanece en memoria y no se imprime ni se escribe dentro del notebook.

Al crear un experimento usaremos esta convención sencilla:

```text
/Shared/<curso>-<periodo>-<actividad>-<caso>
```

El experimento guiado de hoy se llamará `/Shared/pcd-otono-2026-clase-12-nyc-taxi`. Todos los runs de la práctica quedarán dentro de ese experimento.


In [ ]:
import os

import mlflow
from dotenv import load_dotenv

load_dotenv(override=True)

if not os.getenv("DATABRICKS_HOST") or not os.getenv("DATABRICKS_TOKEN"):
    raise ValueError("Faltan DATABRICKS_HOST o DATABRICKS_TOKEN en .env")


In [ ]:

TRACKING_URI = "databricks"
EXPERIMENTO = "/Shared/pcd-otono-2026-clase-12-nyc-taxi"

mlflow.set_tracking_uri(TRACKING_URI)
experimento = mlflow.set_experiment(EXPERIMENTO)

{
    "tracking_uri": mlflow.get_tracking_uri(),
    "experiment_id": experimento.experiment_id,
    "experiment_name": experimento.name,
}


### ✅ Checkpoint: destino remoto

Antes de entrenar, comprueba que:

- la Tracking URI es `databricks`;
- el experimento termina en `clase-12-nyc-taxi`;
- `.env` está ignorado por Git;
- no aparece el host ni el token en la celda o en la salida;
- el experimento es visible en **Experiments** de Databricks.

Si la conexión falla, revisa los nombres `DATABRICKS_HOST` y `DATABRICKS_TOKEN`, la URL base y la vigencia del token. No pegues el token en el notebook.


# 5. Preparar el caso Green Taxi

Reutilizaremos las muestras pequeñas de las clases 10 y 11. Marzo contiene 1,200 filas y abril 500. Los archivos proceden de los Parquet oficiales de [NYC TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page).

Para hacer tuning sin consultar abril repetidamente:

1. dividiremos marzo en entrenamiento interno y validación de tuning;
2. elegiremos hiperparámetros con esa validación interna;
3. reentrenaremos el mejor pipeline con todo marzo;
4. evaluaremos una sola vez con abril.


In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split


def encontrar_raiz() -> Path:
    # Encuentra la raíz del repositorio desde Jupyter o VS Code.
    actual = Path.cwd().resolve()
    for candidata in (actual, *actual.parents):
        if (candidata / "pyproject.toml").exists():
            return candidata
    raise FileNotFoundError("No se encontró la raíz con pyproject.toml")


RAIZ = encontrar_raiz()
DATOS = RAIZ / "labs" / "starters" / "clase-10-experiment-tracking" / "datos"

FEATURES_NUMERICAS = ["distancia_km", "pasajeros", "hora_recoleccion"]
FEATURES_CATEGORICAS = ["zona_origen", "zona_destino"]
FEATURES = FEATURES_NUMERICAS + FEATURES_CATEGORICAS
TARGET = "duracion_minutos"


def cargar_muestra(ruta: Path) -> pd.DataFrame:
    viajes = pd.read_csv(ruta)
    enteras = ["pasajeros", "hora_recoleccion", "zona_origen", "zona_destino"]
    viajes[enteras] = viajes[enteras].astype(int)
    return viajes[FEATURES + [TARGET]]


marzo = cargar_muestra(DATOS / "green-taxi-train.csv")
abril = cargar_muestra(DATOS / "green-taxi-validation.csv")

marzo_train, marzo_tuning = train_test_split(
    marzo,
    test_size=0.2,
    random_state=42,
)

{
    "marzo_train": marzo_train.shape,
    "marzo_tuning": marzo_tuning.shape,
    "abril_final": abril.shape,
    "features": FEATURES,
    "target": TARGET,
}


## 5.1 Registrar la procedencia de los datos

Los objetos `Dataset` describen qué datos utilizó el experimento. Los contextos distinguen entrenamiento interno, validación para tuning y validación final.


In [ ]:
dataset_marzo_train = mlflow.data.from_pandas(
    marzo_train,
    source="green-taxi-train.csv",
    targets=TARGET,
    name="green-taxi-2026-03-train-interno",
)
dataset_marzo_tuning = mlflow.data.from_pandas(
    marzo_tuning,
    source="green-taxi-train.csv",
    targets=TARGET,
    name="green-taxi-2026-03-validacion-tuning",
)
dataset_marzo_completo = mlflow.data.from_pandas(
    marzo,
    source="green-taxi-train.csv",
    targets=TARGET,
    name="green-taxi-2026-03-completo",
)
dataset_abril = mlflow.data.from_pandas(
    abril,
    source="green-taxi-validation.csv",
    targets=TARGET,
    name="green-taxi-2026-04-final",
)


# 6. Tuning de hiperparámetros

Partimos de una pregunta concreta: **¿qué configuración funciona mejor para Linear Regression y para Random Forest cuando conservamos los mismos datos, features, preprocesamiento y métrica?**

Un hiperparámetro se fija antes de entrenar. Linear Regression tiene pocas decisiones útiles para este ejemplo: compararemos `fit_intercept=True` y `False`. En Random Forest ajustaremos `max_depth` y `min_samples_leaf`; el resto permanecerá igual que en las clases anteriores.

La búsqueda seguirá esta secuencia:

1. el **espacio de búsqueda** define qué valores están permitidos;
2. un **trial** toma una combinación concreta;
3. la **función objetivo** construye un pipeline nuevo, lo entrena y devuelve un RMSE;
4. el **study** reúne los trials y conserva el menor RMSE porque indicaremos `direction="minimize"`;
5. el **sampler** decide qué combinación se prueba después;
6. MLflow registra el study como parent run y cada trial como child run.

La función objetivo debe devolver un solo número porque Optuna necesita comparar todos los intentos con el mismo criterio. Abril no aparece dentro de esa función: se reserva para la comparación de los pipelines finales.

Consultas: [Optuna — First Optimization](https://optuna.readthedocs.io/en/stable/tutorial/10_key_features/001_first.html) y [MLflow Hyperparameter Tuning](https://mlflow.org/docs/latest/ml/getting-started/hyperparameter-tuning/).


## 6.1 Partir del pipeline conocido

Mantendremos `construir_pipeline(nombre_modelo)`, la misma interfaz de la Clase 10. La definición queda visible para recordar el preprocesamiento y el `if` que selecciona el estimador. Cada study crea un pipeline nuevo y cambia sólo los parámetros del modelo mediante `set_params()`.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


def construir_pipeline(nombre_modelo: str) -> Pipeline:
    preprocesamiento = ColumnTransformer(
        [
            ("numericas", "passthrough", FEATURES_NUMERICAS),
            (
                "categoricas",
                OneHotEncoder(handle_unknown="ignore"),
                FEATURES_CATEGORICAS,
            ),
        ]
    )

    if nombre_modelo == "linear_regression":
        estimador = LinearRegression()
    elif nombre_modelo == "random_forest":
        estimador = RandomForestRegressor(
            n_estimators=100,
            max_depth=12,
            min_samples_leaf=5,
            n_jobs=-1,
            random_state=42,
        )
    else:
        raise ValueError(f"Modelo no reconocido: {nombre_modelo}")

    return Pipeline(
        [("preprocesamiento", preprocesamiento), ("modelo", estimador)]
    )


pipeline_lineal = construir_pipeline("linear_regression")
pipeline_bosque = construir_pipeline("random_forest")

pipeline_lineal, pipeline_bosque


## 6.2 Ajustar Linear Regression

Linear Regression no tiene un espacio amplio de hiperparámetros. Compararemos de manera exhaustiva las dos opciones de `fit_intercept`: estimar el intercepto o forzarlo a cero. `GridSampler` garantiza que cada opción se pruebe una vez. El menor RMSE interno determina la configuración final; abril permanece fuera del study.


In [ ]:
from time import perf_counter

import mlflow.sklearn
import optuna
from mlflow.models import infer_signature
from optuna.samplers import GridSampler
from sklearn.metrics import root_mean_squared_error


def objective_lineal(trial: optuna.trial.Trial) -> float:
    fit_intercept = trial.suggest_categorical(
        "fit_intercept", [True, False]
    )
    pipeline = construir_pipeline("linear_regression")
    pipeline.set_params(modelo__fit_intercept=fit_intercept)

    inicio = perf_counter()
    pipeline.fit(marzo_train[FEATURES], marzo_train[TARGET])
    tiempo = perf_counter() - inicio
    predicciones = pipeline.predict(marzo_tuning[FEATURES])
    rmse = root_mean_squared_error(marzo_tuning[TARGET], predicciones)

    with mlflow.start_run(run_name=f"trial-{trial.number:03d}", nested=True):
        mlflow.log_param("fit_intercept", fit_intercept)
        mlflow.log_metrics(
            {"tuning_rmse": rmse, "training_time_seconds": tiempo}
        )
        mlflow.set_tags(
            {
                "course": "proyecto-ciencia-datos",
                "term": "otono-2026",
                "class": "12",
                "role": "tuning-trial",
                "model_family": "linear_regression",
            }
        )
    return rmse


study_lineal = optuna.create_study(
    direction="minimize",
    sampler=GridSampler({"fit_intercept": [True, False]}),
)
resultado_lineal = {}

with mlflow.start_run(run_name="tuning-linear-regression") as parent_lineal:
    mlflow.log_input(dataset_marzo_train, context="training")
    mlflow.log_input(dataset_marzo_tuning, context="tuning-validation")
    mlflow.log_params({"optimizer": "optuna-grid", "n_trials": 2})
    mlflow.set_tags(
        {
            "course": "proyecto-ciencia-datos",
            "term": "otono-2026",
            "class": "12",
            "role": "tuning-study",
            "model_family": "linear_regression",
        }
    )
    study_lineal.optimize(objective_lineal, n_trials=2)

    mejores_parametros = dict(study_lineal.best_trial.params)
    mlflow.log_param("best_fit_intercept", mejores_parametros["fit_intercept"])
    mlflow.log_metric("best_tuning_rmse", study_lineal.best_value)

    pipeline_final = construir_pipeline("linear_regression")
    pipeline_final.set_params(
        modelo__fit_intercept=mejores_parametros["fit_intercept"]
    )
    inicio = perf_counter()
    pipeline_final.fit(marzo[FEATURES], marzo[TARGET])
    tiempo_final = perf_counter() - inicio
    predicciones = pipeline_final.predict(abril[FEATURES])
    rmse_final = root_mean_squared_error(abril[TARGET], predicciones)
    ejemplo = abril[FEATURES].head(5)

    with mlflow.start_run(
        run_name="linear-regression-final", nested=True
    ) as final_lineal:
        mlflow.log_input(dataset_marzo_completo, context="training")
        mlflow.log_input(dataset_abril, context="final-validation")
        mlflow.log_params(mejores_parametros)
        mlflow.log_metrics(
            {
                "validation_rmse": rmse_final,
                "training_time_seconds": tiempo_final,
            }
        )
        mlflow.set_tags(
            {
                "course": "proyecto-ciencia-datos",
                "term": "otono-2026",
                "class": "12",
                "role": "model-final",
                "model_family": "linear_regression",
            }
        )
        modelo_lineal = mlflow.sklearn.log_model(
            pipeline_final,
            name="model",
            input_example=ejemplo,
            signature=infer_signature(ejemplo, pipeline_final.predict(ejemplo)),
            serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_PICKLE,
        )
        resultado_lineal.update(
            {
                "parent_run_id": parent_lineal.info.run_id,
                "final_run_id": final_lineal.info.run_id,
                "model_family": "linear_regression",
                "model_id": modelo_lineal.model_id,
                "validation_rmse": rmse_final,
                "training_time_seconds": tiempo_final,
                "best_params": mejores_parametros,
            }
        )

resultado_lineal


## 6.3 Ajustar Random Forest

El segundo study conserva el bosque de las clases anteriores y explora `max_depth` y `min_samples_leaf`. Optuna llamará la función objetivo una vez por trial. En cada llamada debe:

1. pedir a `trial` un valor de `max_depth` y otro de `min_samples_leaf`;
2. crear un pipeline nuevo con `construir_pipeline("random_forest")`;
3. aplicar los valores con `pipeline.set_params(modelo__...)`;
4. entrenar con `marzo_train`;
5. calcular RMSE con `marzo_tuning`;
6. registrar parámetros y métricas en un child run;
7. devolver el RMSE.

Se crea un pipeline nuevo para que cada trial comience sin estado entrenado. La función devuelve un solo número porque el study necesita comparar todos los intentos con el mismo criterio. En los trials guardaremos parámetros y métricas; el modelo completo se registra sólo al terminar el study.


In [ ]:
def objective_bosque(trial: optuna.trial.Trial) -> float:
    parametros = {
        "max_depth": trial.suggest_int("max_depth", 6, 18, step=2),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 2, 10, step=2),
    }
    pipeline = construir_pipeline("random_forest")
    pipeline.set_params(
        modelo__max_depth=parametros["max_depth"],
        modelo__min_samples_leaf=parametros["min_samples_leaf"],
    )

    inicio = perf_counter()
    pipeline.fit(marzo_train[FEATURES], marzo_train[TARGET])
    tiempo = perf_counter() - inicio
    predicciones = pipeline.predict(marzo_tuning[FEATURES])
    rmse = root_mean_squared_error(marzo_tuning[TARGET], predicciones)

    with mlflow.start_run(
        run_name=f"trial-{trial.number:03d}",
        nested=True,
    ) as child_run:
        mlflow.log_params(parametros)
        mlflow.log_metrics(
            {
                "tuning_rmse": rmse,
                "training_time_seconds": tiempo,
            }
        )
        mlflow.set_tags(
            {
                "course": "proyecto-ciencia-datos",
                "term": "otono-2026",
                "class": "12",
                "role": "tuning-trial",
                "model_family": "random_forest",
                "trial_number": str(trial.number),
            }
        )
        trial.set_user_attr("run_id", child_run.info.run_id)

    return rmse


## 6.4 Ejecutar el study de Random Forest

Usaremos ocho trials: los primeros tres son la exploración inicial de TPE y los siguientes pueden aprovechar lo observado. La semilla hace reproducible la secuencia.

El parent run registra la pregunta general. Cada child run conserva un intento y el parent resume el mejor resultado. Esta búsqueda es didáctica: ocho trials no garantizan encontrar una configuración globalmente óptima.

Consulta: [TPESampler](https://optuna.readthedocs.io/en/stable/reference/samplers/generated/optuna.samplers.TPESampler.html).


In [ ]:
from optuna.samplers import TPESampler

N_TRIALS = 8
sampler = TPESampler(seed=42, n_startup_trials=3)
study_bosque = optuna.create_study(direction="minimize", sampler=sampler)
resultado_bosque = {}

with mlflow.start_run(run_name="tuning-random-forest") as parent_run:
    mlflow.log_input(dataset_marzo_train, context="training")
    mlflow.log_input(dataset_marzo_tuning, context="tuning-validation")
    mlflow.log_params(
        {
            "optimizer": "optuna-tpe",
            "n_trials": N_TRIALS,
            "sampler_seed": 42,
            "sampler_startup_trials": 3,
            "objective_metric": "tuning_rmse",
        }
    )
    mlflow.set_tags(
        {
            "course": "proyecto-ciencia-datos",
            "term": "otono-2026",
            "class": "12",
            "role": "tuning-study",
            "model_family": "random_forest",
        }
    )

    study_bosque.optimize(objective_bosque, n_trials=N_TRIALS)

    mejores_parametros = dict(study_bosque.best_trial.params)
    mlflow.log_params(
        {f"best_{clave}": valor for clave, valor in mejores_parametros.items()}
    )
    mlflow.log_metrics({"best_tuning_rmse": study_bosque.best_value})
    mlflow.log_param("best_trial_number", study_bosque.best_trial.number)

    pipeline_final = construir_pipeline("random_forest")
    pipeline_final.set_params(
        modelo__max_depth=mejores_parametros["max_depth"],
        modelo__min_samples_leaf=mejores_parametros["min_samples_leaf"],
    )
    parametros_bosque = {
        "n_estimators": pipeline_final.get_params()["modelo__n_estimators"],
        **mejores_parametros,
    }
    inicio = perf_counter()
    pipeline_final.fit(marzo[FEATURES], marzo[TARGET])
    tiempo_final = perf_counter() - inicio
    predicciones = pipeline_final.predict(abril[FEATURES])
    rmse_final = root_mean_squared_error(abril[TARGET], predicciones)

    ejemplo = abril[FEATURES].head(5)
    firma = infer_signature(ejemplo, pipeline_final.predict(ejemplo))

    with mlflow.start_run(run_name="random-forest-final", nested=True) as final_run:
        mlflow.log_input(dataset_marzo_completo, context="training")
        mlflow.log_input(dataset_abril, context="final-validation")
        mlflow.log_params(parametros_bosque)
        mlflow.log_metrics(
            {
                "validation_rmse": rmse_final,
                "training_time_seconds": tiempo_final,
            }
        )
        mlflow.set_tags(
            {
                "course": "proyecto-ciencia-datos",
                "term": "otono-2026",
                "class": "12",
                "role": "model-final",
                "model_family": "random_forest",
                "best_trial_number": str(study_bosque.best_trial.number),
            }
        )
        modelo_bosque = mlflow.sklearn.log_model(
            pipeline_final,
            name="model",
            input_example=ejemplo,
            signature=firma,
            serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_PICKLE,
        )
        resultado_bosque.update(
            {
                "parent_run_id": parent_run.info.run_id,
                "final_run_id": final_run.info.run_id,
                "model_id": modelo_bosque.model_id,
                "model_family": "random_forest",
                "validation_rmse": rmse_final,
                "training_time_seconds": tiempo_final,
                "best_params": parametros_bosque,
            }
        )

resultado_bosque


### ✅ Checkpoint: parent y child runs

En Databricks, abre el experimento `/Shared/pcd-otono-2026-clase-12-nyc-taxi` y comprueba:

- un parent run `tuning-linear-regression`;
- dos child runs lineales y `linear-regression-final`;
- un parent run `tuning-random-forest`;
- ocho child runs llamados `trial-000` a `trial-007`;
- un child run `random-forest-final`;
- parámetros y `tuning_rmse` en cada trial;
- `validation_rmse` sólo en los modelos evaluados con abril;
- parámetros y métricas en cada trial, sin duplicar un modelo completo por intento;
- Logged Models únicamente para los dos modelos finales.

![Captura histórica de la interfaz de MLflow utilizada para comparar varios runs.](../assets/modulo-02-ciclo-mlops/clase-10/mlflow-run-comparison.png)

La ubicación de botones y columnas puede cambiar, pero experiments, runs, parámetros, métricas, datasets y modelos cumplen las mismas funciones.


# 7. Comparar los dos modelos ajustados

El RMSE interno sirvió para elegir hiperparámetros dentro de cada familia. Ahora usamos abril para comparar Linear Regression y Random Forest bajo las mismas condiciones. Sólo después de ordenar esta tabla asignaremos aliases.


In [ ]:
comparacion = pd.DataFrame(
    [
        resultado_lineal,
        resultado_bosque,
    ]
).sort_values("validation_rmse").reset_index(drop=True)

comparacion


La tabla permite preguntar:

1. ¿qué familia obtuvo el menor RMSE de abril?;
2. ¿la diferencia es relevante en minutos?;
3. ¿cuánto cambió el tiempo de entrenamiento?;
4. ¿la configuración es más compleja?;
5. ¿la diferencia justifica preferir el modelo más complejo?

La comparación determina los aliases: la primera fila será `champion` y la segunda `challenger`. Los aliases describen el ranking observado; no pertenecen permanentemente a una familia y no despliegan el modelo.


# 8. Registrar los modelos en Unity Catalog

![Diagrama histórico que conecta los modelos de los runs con versiones administradas por Model Registry.](../assets/modulo-02-ciclo-mlops/clase-11/mlflow-tracking-server-model-registry.png)

MLflow 3 asigna un `model_id` a cada Logged Model. Podemos registrar directamente esa identidad mediante una URI `models:/<model_id>`. En Unity Catalog el nombre tiene tres partes: `catalogo.esquema.modelo`.

Free Edition normalmente permite trabajar en el catálogo visible en **Catalog Explorer** y en el esquema `default`. Escribe el nombre exacto; no uses el identificador de la URL del workspace.

Consulta: [Log, load and register MLflow models](https://docs.databricks.com/aws/en/mlflow/models) y [MLflow Model Registry workflow](https://mlflow.org/docs/latest/ml/model-registry/workflow/).


In [ ]:
REGISTRY_URI = "databricks-uc"
CATALOGO = input("Nombre exacto del catálogo del workspace: " ).strip()
ESQUEMA = "default"
if not CATALOGO:
    raise ValueError("Consulta Catalog Explorer y escribe el nombre del catálogo")
MODELO_REGISTRADO = f"{CATALOGO}.{ESQUEMA}.nyc_taxi_trip_duration"
mlflow.set_registry_uri(REGISTRY_URI)

version_lineal = mlflow.register_model(
    model_uri=f"models:/{resultado_lineal['model_id']}",
    name=MODELO_REGISTRADO,
)
version_bosque = mlflow.register_model(
    model_uri=f"models:/{resultado_bosque['model_id']}",
    name=MODELO_REGISTRADO,
)

versiones = {
    "linear_regression": version_lineal,
    "random_forest": version_bosque,
}
versiones


## 8.1 Asignar aliases y tags

Los aliases son punteros mutables. Las versiones no cambian; el alias puede apuntar a otra versión después de una decisión de promoción.


In [ ]:
from mlflow import MlflowClient

client = MlflowClient(tracking_uri=TRACKING_URI, registry_uri=REGISTRY_URI)

familia_champion = comparacion.loc[0, "model_family"]
familia_challenger = comparacion.loc[1, "model_family"]
version_champion = versiones[familia_champion]
version_challenger = versiones[familia_challenger]

client.set_registered_model_alias(
    name=MODELO_REGISTRADO,
    alias="champion",
    version=version_champion.version,
)
client.set_registered_model_alias(
    name=MODELO_REGISTRADO,
    alias="challenger",
    version=version_challenger.version,
)

client.set_model_version_tag(
    MODELO_REGISTRADO,
    version_champion.version,
    "validation_rmse",
    f"{comparacion.loc[0, 'validation_rmse']:.6f}",
)
client.set_model_version_tag(
    MODELO_REGISTRADO,
    version_challenger.version,
    "validation_rmse",
    f"{comparacion.loc[1, 'validation_rmse']:.6f}",
)


## 8.2 Cargar ambos aliases

La URI conserva la misma idea de la Clase 11, pero ahora el nombre incluye catálogo, esquema y modelo.


In [ ]:
champion_uri = f"models:/{MODELO_REGISTRADO}@champion"
challenger_uri = f"models:/{MODELO_REGISTRADO}@challenger"

modelo_champion = mlflow.pyfunc.load_model(champion_uri)
modelo_challenger = mlflow.pyfunc.load_model(challenger_uri)

muestra = abril[FEATURES].head(5)
pd.DataFrame(
    {
        "champion": modelo_champion.predict(muestra),
        "challenger": modelo_challenger.predict(muestra),
    }
)


# 9. Errores frecuentes

| Síntoma | Causa probable | Siguiente verificación |
|---|---|---|
| El run aparece localmente | Tracking URI incorrecta | imprime `mlflow.get_tracking_uri()` |
| Faltan variables de Databricks | `.env` no está en la raíz o los nombres no coinciden | revisa `DATABRICKS_HOST` y `DATABRICKS_TOKEN` sin imprimir sus valores |
| Databricks responde `401` | token vencido, revocado o mal copiado | genera otro token y reemplázalo en `.env` |
| Todos los trials quedan al mismo nivel | falta `nested=True` | revisa `mlflow.start_run` dentro de `objective` |
| El RMSE de abril cambia durante el tuning | abril se usó dentro de `objective` | usa únicamente `marzo_tuning` en los trials |
| Registry rechaza el nombre | falta catálogo o esquema | confirma `catalogo.esquema.modelo` |
| No se puede registrar | falta permiso en Unity Catalog | revisa Catalog Explorer y el esquema elegido |
| Aparecen demasiados artifacts | se guardó un modelo en muchos trials | reduce trials o guarda sólo el modelo final |
| El modelo cargado pide columnas transformadas | se registró sólo el estimador | registra el `Pipeline` completo |
| Una mejora mínima se considera suficiente | se miró sólo RMSE | compara tiempo, complejidad y magnitud de la mejora |

> **Seguridad:** esta práctica conserva serialización `pickle` por compatibilidad con el pipeline validado del curso. Carga únicamente artifacts propios o provenientes de un workspace confiable. MLflow ya ofrece formatos más seguros, pero siguen evolucionando y no forman parte del objetivo de esta sesión. Consulta [Pickle-Free Model format](https://mlflow.org/docs/latest/ml/tracking/pickle-free-models).


# 10. Recopilación

- Free Edition ofrece un workspace individual, serverless y sujeto a cuotas para aprender sin contratar una nube.
- La URL base identifica el workspace; el catálogo se identifica por separado en Catalog Explorer.
- `.env` conserva `DATABRICKS_HOST` y `DATABRICKS_TOKEN` fuera de Git.
- El SDK puede usar almacenamiento local, una base local o un Tracking Server remoto.
- Databricks administra el servidor; nuestro código sigue definiendo datos, pipeline y criterio de comparación.
- Un study de Optuna reúne trials que responden la misma pregunta.
- Linear Regression usa una búsqueda exhaustiva pequeña y Random Forest usa TPE.
- Los child runs evitan mezclar los intentos y conservan sus parámetros y métricas.
- Marzo se usa para tuning; abril queda reservado para comparar modelos finales.
- Un Logged Model tiene identidad propia en MLflow 3.
- Unity Catalog organiza versiones bajo un nombre de tres niveles.
- `champion` y `challenger` son aliases; registrar no equivale a desplegar.


# ✅ Check final de la clase

- [ ] Creé o recuperé una cuenta individual de Databricks Free Edition.
- [ ] Distingo workspace, URL base, catálogo y esquema.
- [ ] El `.env` contiene las variables requeridas y permanece fuera de Git.
- [ ] Distingo las tres configuraciones comunes de MLflow.
- [ ] Puedo explicar Backend Store y Artifact Store.
- [ ] El experimento exclusivo aparece en mi workspace de Databricks.
- [ ] Existe un parent lineal con dos trials y un modelo final.
- [ ] Existe un parent de bosque con ocho trials y un modelo final.
- [ ] Abril no se utilizó dentro de la función objetivo.
- [ ] El mejor trial se obtuvo programáticamente.
- [ ] Ambos modelos finales se reentrenaron con marzo completo.
- [ ] Puedo explicar qué reciben y qué devuelven las funciones objetivo.
- [ ] Champion y challenger tienen versiones y aliases en Unity Catalog.
- [ ] Puedo cargar ambos aliases y producir predicciones.
- [ ] Puedo justificar por qué una familia quedó como champion.

Este check detecta dudas; no es una entrega.


# 11. Tarea 5 — Ajustar y comparar tres modelos

**Entrega:** lunes 5 de octubre de 2026 a las **19:55**, antes de la clase.

La tarea continúa el caso NYC Taxi de la Tarea 3. Trabajarás en tu repositorio privado individual `pcd-entregas-2026`; no es una extensión de este notebook ni se realiza en el repositorio público.

## Punto de partida

Conserva de la Tarea 3:

- las mismas cinco features y `duracion_minutos` como target;
- el mismo `ColumnTransformer`;
- Linear Regression y Random Forest como familias conocidas;
- marzo para desarrollo, abril para validación final y RMSE en minutos.

No subas los pickles anteriores. Vuelve a entrenar los modelos dentro del nuevo experimento para que sus datos, parámetros y métricas queden registrados.

## Lo que debes implementar

1. Crea el experimento exclusivo `/Shared/pcd-otono-2026-tarea-05-nyc-taxi` en el servidor de MLflow de tu cuenta de Databricks.
2. Ajusta Linear Regression en `tuning-linear-regression` con dos trials exhaustivos.
3. Ajusta Random Forest en `tuning-random-forest` con 6–8 trials.
4. Ajusta Gradient Boosting en `tuning-gradient-boosting` con 6–8 trials.
5. Usa solamente las particiones internas de marzo dentro de las funciones objetivo.
6. Reentrena el mejor modelo de cada familia con marzo completo y evalúa los tres una sola vez con abril.
7. Registra los tres modelos finales en Unity Catalog y asigna `champion`, `challenger` y `candidate` según el ranking.

El experimento de la tarea es distinto al experimento de clase. Conserva `DATABRICKS_HOST` y `DATABRICKS_TOKEN` en el `.env` de la raíz y nunca los agregues a Git.

## Entrega

La carpeta `tareas/tarea-05-tuning-mlflow-databricks/` debe incluir `ajustar_modelos.py`, un README con la comparación y la decisión, y tres evidencias: runs anidados, comparación con Registry y carga de los tres aliases. Trabaja en la rama `feat/05-tuning-mlflow-databricks`, fusiona el pull request y entrega su URL en Canvas.

Consulta los requisitos, evidencias y rúbrica completos en [Tarea 5 — Comparar tres familias con HPO y MLflow](../docs/tareas/tarea-05-tuning-mlflow-databricks.md).


# Referencias y consultas

## MLflow

### Consulta rápida de operaciones

![Operaciones frecuentes de MLflow para registrar y consultar parámetros, métricas, artifacts, texto e imágenes.](../assets/modulo-02-ciclo-mlops/clase-12/mlflow-cheatsheet.png)

Usa esta imagen para ubicar la operación que necesitas y confirma su firma en la [API actual de MLflow](https://mlflow.org/docs/latest/api_reference/python_api/mlflow.html).

- [Architecture Overview](https://mlflow.org/docs/latest/self-hosting/architecture/overview/)
- [Tracking Server](https://mlflow.org/docs/latest/self-hosting/architecture/tracking-server/)
- [Tracking Hyperparameter Tuning](https://mlflow.org/docs/latest/ml/getting-started/hyperparameter-tuning/)
- [Parent and Child Runs](https://mlflow.org/docs/latest/ml/traditional-ml/tutorials/hyperparameter-tuning/part1-child-runs/)
- [Search Logged Models](https://mlflow.org/docs/latest/ml/search/search-models/)
- [Model Registry Workflow](https://mlflow.org/docs/latest/ml/model-registry/workflow/)
- [Pickle-Free Model Format](https://mlflow.org/docs/latest/ml/tracking/pickle-free-models)

## Databricks

- [Comparar Free Edition y free trial](https://docs.databricks.com/aws/en/getting-started/free-trial-vs-free-edition)
- [Crear una cuenta de Databricks Free Edition](https://docs.databricks.com/aws/en/getting-started/free-edition)
- [Página de registro de Free Edition](https://login.databricks.com/signup?provider=DB_FREE_TIER)
- [Free Edition limitations](https://docs.databricks.com/aws/en/getting-started/free-edition-limitations)
- [Personal access tokens](https://docs.databricks.com/aws/en/dev-tools/auth/pat)
- [Authentication environment variables](https://docs.databricks.com/aws/en/dev-tools/auth/env-vars)
- [MLflow on Databricks](https://docs.databricks.com/aws/en/mlflow)
- [Databricks-hosted Tracking Server](https://docs.databricks.com/aws/en/mlflow/tracking-server-configuration)
- [Log, load and register MLflow models](https://docs.databricks.com/aws/en/mlflow/models)
- [python-dotenv](https://bbc2.github.io/python-dotenv/)

## Optuna

- [Optuna documentation](https://optuna.readthedocs.io/en/stable/)
- [Optuna tutorial](https://optuna.readthedocs.io/en/stable/tutorial/index.html)
